# Fundamental SPP — GPU sweep on Colab

End-to-end reproduction of *Fundamental Information for Low-Turnover Equity ML*.
This notebook resumes the 60-config walk-forward sweep (4 architectures x 3 regimes x 5 years)
on a Colab GPU, picking up where the CPU box left off (8/60 configs already done).

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU** (T4 is fine).

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Set Runtime -> Change runtime type -> GPU.'
!nvidia-smi -L

## 1. Get the code

In [ ]:
import os
if not os.path.exists('Fundamental_SPP'):
    !git clone https://github.com/ShadowKingYT444/Fundamental_SPP
else:
    !git -C Fundamental_SPP pull --ff-only
%cd Fundamental_SPP
!git log --oneline -1

## 2. Dependencies
Colab already ships torch (CUDA), pandas, numpy, scipy, scikit-learn and matplotlib. Only pyarrow (parquet) may be missing.

In [ ]:
!pip install -q pyarrow

## 3. Stage the data from Google Drive
The ~520 MB prepared dataset (panels, prices) plus the 8 finished checkpoints/scores were uploaded to Drive beforehand. This mounts Drive and copies them into the repo layout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, glob
SRC = '/content/drive/MyDrive/fundamental_spp_data'
assert os.path.isdir(SRC), f'Data folder not found: {SRC}'
for d in ['data', 'checkpoints', 'scores', 'results']:
    os.makedirs(d, exist_ok=True)
n = 0
for p in sorted(glob.glob(os.path.join(SRC, '*'))):
    base = os.path.basename(p)
    if base.endswith('.parquet'): dst = os.path.join('data', base)
    elif base.endswith('.pt'): dst = os.path.join('checkpoints', base)
    elif base.endswith('.csv'): dst = os.path.join('scores', base)
    elif base == 'run_status.json': dst = os.path.join('results', base)
    else: continue
    if not os.path.exists(dst):
        shutil.copy(p, dst); n += 1
print(f'staged {n} new files')
!ls -lh data/ | head -12

## 4. Smoke test: 1 epoch of MISS on the GPU
Uses a throwaway checkpoint dir so the real sweep state is untouched. Watch for `device=cuda` in the log and a sensible per-batch time.

In [ ]:
!python -u src/run_train.py --model miss --regime tech63 --year 2021 --epochs 1 --device cuda --checkpoint-dir /tmp/smoke_ckpt --seed 123 2>&1 | tail -6

## 5. Run the sweep (resumable)
Picks up at 8/60: the two blocked long configs (`miss_tech63_2024/2025`) run at full batch 512 on the GPU, then everything else. Fully resumable — if Colab disconnects, just re-run this cell. Budget is ~11h; Colab caps sessions around 12h.

In [ ]:
!python -u src/run_next.py --device cuda --allow-long --time-budget 40000

## 6. Evaluate + figures (after 60/60)
Once the sweep prints DONE, build the metrics, tables and paper figures.

In [ ]:
!python -u src/evaluate.py 2>&1 | tail -20
!python -u src/make_figures.py 2>&1 | tail -5

## 7. Sync everything back to Drive
Checkpoints, scores, metrics and figures land in `fundamental_spp_data/colab_out/` for download.

In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/fundamental_spp_data/colab_out'
os.makedirs(OUT, exist_ok=True)
for d in ['checkpoints', 'scores', 'results']:
    dst = os.path.join(OUT, d)
    if os.path.exists(dst): shutil.rmtree(dst)
    if os.path.isdir(d): shutil.copytree(d, dst)
print('synced to', OUT)
!du -sh /content/drive/MyDrive/fundamental_spp_data/colab_out